# Computer Vision for Urban Research: Hands-on in Colab

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/BayiLi081/GIS-training/blob/gh-pages/cvurban_colab.ipynb)

We use the lecture's **Singapore traffic camera sample** (8 geotagged images, Singapore Open Data Licence v1.0) to:

1. look at the images and their locations
2. **detect** vehicles with two models (RT-DETR and YOLO), compare them, and choose one
3. **segment** every pixel into classes such as road, vegetation, sky (semantic segmentation)
4. turn the pixels into **indicators** (vehicle count, green view share) and **map** them

Everything runs in the cloud: no installation on your laptop.
Optional speed-up: *Runtime → Change runtime type → T4 GPU*. The CPU runtime also works; it just takes a minute longer.

## 1. Download the dataset

In [ ]:
import urllib.request
from pathlib import Path

import pandas as pd

BASE = "https://raw.githubusercontent.com/BayiLi081/GIS-training/gh-pages/data/sg_traffic_images/"

meta = pd.read_csv(BASE + "metadata.csv")
Path("images").mkdir(exist_ok=True)
for f in meta["file"]:
    if not Path(f).exists():
        urllib.request.urlretrieve(BASE + f, f)

meta[["camera_id", "latitude", "longitude", "timestamp"]]

## 2. Look at the images

Each image is a raster: 360 rows × 640 columns × 3 colour bands (RGB).

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

images = {row.camera_id: Image.open(row.file).convert("RGB") for row in meta.itertuples()}

fig, axes = plt.subplots(2, 4, figsize=(18, 5.5))
for ax, (cam, img) in zip(axes.flat, images.items()):
    ax.imshow(img)
    ax.set_title(f"camera {cam}")
    ax.axis("off")
plt.tight_layout()

## 3. Object detection: where are the vehicles? Compare two models

An **object detector** takes an image and returns a list of objects. Each object has:

- a **box**: pixel coordinates `xmin, ymin, xmax, ymax`
- a **label**: one of the classes the model was trained on. Both models here were trained on **COCO**, a dataset of everyday photos with 80 classes, including car, bus, truck, motorcycle and person.
- a **confidence score** between 0 and 1. We keep only boxes above a chosen threshold.

Many free pretrained detectors exist, and they do not give the same answer. We use two that solve the same task in quite different ways.

### YOLO: one quick look, a prediction everywhere

**YOLO** ("You Only Look Once", Redmon et al., 2016) treats detection as a single prediction step:

1. A convolutional neural network (**CNN**) turns the image into feature maps at three scales: coarse maps for large vehicles, fine maps for small ones.
2. **Every location** on those maps predicts a box, a label and a score, so one pass over the image proposes many candidate boxes.
3. Duplicate boxes for the same object are removed. Older YOLO versions do this with a separate step called *non-maximum suppression* (NMS); **YOLO26**, used here, is trained to output one box per object directly.

> image → CNN → a prediction at every grid location → remove duplicates → boxes

**In practice:** fast and light. It runs on a laptop CPU or on a device next to the camera, and the Ultralytics toolkit can also train, track and export it. Small or tightly packed objects can be missed, or merged into one box.

### RT-DETR: a fixed set of questions about the whole image

**RT-DETR** (Real-Time DEtection TRansformer; Zhao et al., 2024; version 2: Lv et al., 2024) builds on **DETR** (Carion et al., 2020). DETR brought the **transformer**, the architecture behind large language models, to object detection:

1. A CNN **backbone** extracts image features. Here it is ResNet-18, hence `r18` in the model name.
2. A transformer **encoder** mixes those features, so each part of the image is described in the context of the rest.
3. A transformer **decoder** starts from **300 object queries**: slots that each ask the whole image for one object. Through *attention*, every query can look anywhere in the image, and each returns one box, label and score.
4. In training, each real object is matched to exactly one query, so no duplicate-removal step is needed.

> image → CNN backbone → transformer encoder → 300 object queries → one box per query

**In practice:** context from the whole image can help tell neighbouring objects apart. It is heavier than a small YOLO on a CPU, and finds at most 300 objects per image (plenty here). It runs through Hugging Face `transformers`, the same library we use for segmentation in section 4.

### The two models at a glance

| | **RT-DETR** (`PekingU/rtdetr_v2_r18vd`) | **YOLO** (`yolo26s.pt`) |
|---|---|---|
| Design | CNN backbone + transformer; 300 object queries | CNN; predicts at every grid location |
| Size | ~20 M parameters | ~10 M parameters |
| Library | Hugging Face `transformers` | `ultralytics` |
| Trained on | COCO: 80 everyday classes | COCO: the same 80 classes |
| Default confidence cut-off | 0.5 (Hugging Face pipeline) | 0.25 (Ultralytics) |
| Licence | Apache-2.0 | AGPL-3.0 (commercial licence available) |

Both use the same COCO classes, so their outputs can be compared directly.

**Before running anything, make a prediction.** From the designs, which model do you expect to be faster? Which might separate vehicles in a dense queue better? Write down your guesses. A design suggests a hypothesis; only a test on *your* images answers it. The aim is not the best model *in general*, but the best option **for these cameras and this question**.

*References:* Redmon et al. (2016), [You Only Look Once](https://arxiv.org/abs/1506.02640) · Carion et al. (2020), [End-to-End Object Detection with Transformers](https://arxiv.org/abs/2005.12872) · Zhao et al. (2024), [DETRs Beat YOLOs on Real-time Object Detection](https://arxiv.org/abs/2304.08069) · Lv et al. (2024), [RT-DETRv2](https://arxiv.org/abs/2407.17140) · [Ultralytics YOLO26 documentation](https://docs.ultralytics.com/models/yolo26/)

Weights download automatically the first time.

In [ ]:
%pip install -q ultralytics

In [ ]:
import time

import torch
from transformers import pipeline
from ultralytics import YOLO

device = 0 if torch.cuda.is_available() else -1
print("Running on", "GPU" if device == 0 else "CPU")

rtdetr = pipeline("object-detection", model="PekingU/rtdetr_v2_r18vd", device=device)
yolo = YOLO("yolo26s.pt")  # also try "yolo26n.pt" (smaller, faster) or "yolo26m.pt" (larger)

# Confidence cut-offs: start from each library's default. Scores are not calibrated
# the same way across models, so 0.5 from one model is not 0.5 from another.
THRESHOLDS = {"RT-DETR": 0.5, "YOLO": 0.25}


def detect_rtdetr(img, threshold):
    return rtdetr(img, threshold=threshold)


def detect_yolo(img, threshold):
    # return YOLO results in the same format as the Hugging Face pipeline
    r = yolo.predict(img, conf=threshold, verbose=False)[0]
    return [{"label": r.names[int(c)], "score": s, "box": dict(zip(["xmin", "ymin", "xmax", "ymax"], b))}
            for b, s, c in zip(r.boxes.xyxy.tolist(), r.boxes.conf.tolist(), r.boxes.cls.tolist())]


DETECTORS = {"RT-DETR": detect_rtdetr, "YOLO": detect_yolo}

In [ ]:
import matplotlib.patches as patches

def show_detections(ax, img, detections, title=""):
    ax.imshow(img)
    for d in detections:
        b = d["box"]
        ax.add_patch(patches.Rectangle((b["xmin"], b["ymin"]), b["xmax"] - b["xmin"], b["ymax"] - b["ymin"],
                                       fill=False, color="yellow", linewidth=1.5))
        ax.text(b["xmin"], b["ymin"] - 2, f"{d['label']} {d['score']:.2f}", color="black", fontsize=7,
                backgroundcolor="yellow")
    ax.set_title(title)
    ax.axis("off")

cam = 4703  # try another camera_id
fig, axes = plt.subplots(1, 2, figsize=(20, 5.8))
for ax, (name, detect) in zip(axes, DETECTORS.items()):
    dets = detect(images[cam], THRESHOLDS[name])
    show_detections(ax, images[cam], dets, f"{name} (threshold {THRESHOLDS[name]}), camera {cam}: {len(dets)} objects")
plt.tight_layout()

Look closely before counting. Which vehicles do **both** models find? Which does only one find? Which labels are wrong?

Now count vehicles in **every** image with both models, and time each one.

In [ ]:
VEHICLES = {"car", "truck", "bus", "motorcycle", "bicycle"}
for name, detect in DETECTORS.items():
    detect(images[cam], THRESHOLDS[name])  # warm-up: the first call includes one-off setup time

rows = []
for cam_id in meta["camera_id"]:
    row = {"camera_id": cam_id}
    for name, detect in DETECTORS.items():
        start = time.perf_counter()
        dets = detect(images[cam_id], THRESHOLDS[name])
        row[f"{name} ms"] = round((time.perf_counter() - start) * 1000)
        row[f"{name} vehicles"] = sum(d["label"] in VEHICLES for d in dets)
    rows.append(row)

compare = pd.DataFrame(rows)
compare["difference"] = compare["YOLO vehicles"] - compare["RT-DETR vehicles"]
compare[["camera_id", "RT-DETR vehicles", "YOLO vehicles", "difference", "RT-DETR ms", "YOLO ms"]]

### Which detector should you use? Test, then decide

The two models disagree, and neither count is the truth. A benchmark score on COCO tells you how a model does on *COCO photos*, not on elevated Singapore traffic cameras. So run a small test of your own:

1. **Count by hand** the vehicles in 3–4 images. Zoom in, and decide first whether queued motorcycles count.
2. **Compare** each model's count with yours (next cell). Then change `THRESHOLDS` and rerun: the cut-off can matter as much as the model.
3. **Weigh** accuracy against what else your project needs:

| Criterion | Question to ask |
|---|---|
| Accuracy on *your* images | Which model is closer to your counts? Which of its errors would change the decision? |
| Speed | The `ms` columns: fast enough for live video, or only for occasional snapshots? |
| Hardware | Does it run on a laptop CPU, or does it need a GPU? |
| Licence | Can you use it where you intend to: a class project, a government study, a commercial product? |
| Ease of use | Installation, documentation, support for tracking or retraining on local images |

Other candidates are one line away: another YOLO size, or any object-detection model on the [Hugging Face Hub](https://huggingface.co/models?pipeline_tag=object-detection). Testing what is available is part of the method.

In [ ]:
# Your manual counts: zoom in on each image and replace None with a number
manual = {4703: None, 2701: None, 4799: None}

truth = pd.Series(manual, dtype="float").dropna()
if truth.empty:
    print("Fill in `manual` with your own counts, then rerun this cell.")
else:
    checked = compare.set_index("camera_id").loc[truth.index]
    for name in DETECTORS:
        error = (checked[f"{name} vehicles"] - truth).abs()
        print(f"{name}: off by {error.mean():.1f} vehicles per image on average ({len(truth)} images checked)")

In [ ]:
CHOSEN = "RT-DETR"  # your decision: "RT-DETR" or "YOLO". Note why in your evidence statement.
meta["vehicles"] = compare[f"{CHOSEN} vehicles"].to_numpy()
meta[["camera_id", "vehicles"]]

## 4. Semantic segmentation: what is every pixel?

We use **SegFormer** trained on **Cityscapes** (street scenes from European cities).
It labels every pixel as one of 19 classes: road, sidewalk, building, vegetation, sky, car, …

In [ ]:
segmenter = pipeline("image-segmentation", model="nvidia/segformer-b0-finetuned-cityscapes-1024-1024", device=device)

In [ ]:
import numpy as np

def show_segmentation(img, segments, title=""):
    label_map = np.zeros((img.height, img.width), dtype=int)
    for i, s in enumerate(segments):
        label_map[np.array(s["mask"]) > 0] = i
    cmap = plt.get_cmap("tab20", 20)
    fig, axes = plt.subplots(1, 2, figsize=(18, 5))
    axes[0].imshow(img)
    axes[0].set_title(title)
    axes[1].imshow(img)
    axes[1].imshow(label_map, cmap=cmap, vmin=0, vmax=19, alpha=0.6, interpolation="nearest")
    axes[1].legend(handles=[patches.Patch(color=cmap(i), label=s["label"]) for i, s in enumerate(segments)],
                   loc="center left", bbox_to_anchor=(1, 0.5))
    for ax in axes:
        ax.axis("off")

cam = 2704  # try another camera_id
segments = segmenter(images[cam])
show_segmentation(images[cam], segments, f"camera {cam}")

## 5. From pixels to indicators

Share of a class = pixels of that class / all pixels. For vegetation this is a simple **green view share**
(the Green View Index idea from the lecture; here from an elevated camera, not a pedestrian's eye level).

In [ ]:
def class_shares(img):
    total = img.width * img.height
    return {s["label"]: (np.array(s["mask"]) > 0).sum() / total for s in segmenter(img)}

shares = pd.DataFrame([class_shares(images[cam]) for cam in meta["camera_id"]]).fillna(0)
for cls in ["vegetation", "sky", "road", "building"]:
    meta[f"{cls}_share"] = shares.get(cls, 0)

meta[["camera_id", "vehicles", "vegetation_share", "sky_share", "road_share", "building_share"]].round(3)

## 6. Map the indicators

Keep the location attached to every measurement: that is what makes it GIS data. Hover over a point to see its values.

In [ ]:
import geopandas as gpd

gdf = gpd.GeoDataFrame(meta, geometry=gpd.points_from_xy(meta["longitude"], meta["latitude"]), crs="EPSG:4326")
gdf.explore(
    column="vegetation_share",
    cmap="Greens",
    tooltip=["camera_id", "vehicles", "vegetation_share", "sky_share", "road_share"],
    marker_kwds={"radius": 12},
    tiles="CartoDB positron",
)

## 7. Questions

1. Look at the detections for camera 4703. Which boxes are wrong in each model (hint: look for `train`)? Why might motorcyclists be counted as `person` rather than `motorcycle`?
2. Which detector did you choose, and what evidence supports that choice? Set both thresholds to 0.5 and rerun the comparison: what happens to YOLO's counts? Why is a confidence score not comparable between models, and does your choice change?
3. Compare the segmentation of camera 2704 with the photo. Which classes look right, and which are confused? Then try camera 2701: what does the model call the sea?
4. These models were trained on other places and viewpoints (COCO photos, European street scenes at eye level). What problems can this **domain shift** cause for elevated Singapore traffic cameras?
5. Is the vehicle count from one snapshot a good measure of **traffic flow**? What extra data would you need?
6. Run `download_sample.py` from the dataset folder (or the data.gov.sg API) at a different time of day. Do the indicators change?
7. Which GIS layer (road segments, planning areas, …) would you join these points to, and what planning question could that answer?

*Data: Contains information from Traffic Images accessed on 18 November 2026 from data.gov.sg which is made available under the terms of the Singapore Open Data Licence version 1.0.*